# Model Development and Evaluation

This notebook focuses on training and evaluating machine learning models for credit default prediction.

The objective is to identify patterns associated with loan repayment behavior and compare multiple algorithms using appropriate classification metrics.

Model performance will primarily be assessed using ROC-AUC due to the imbalanced nature of the target variable.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    cross_val_score
)

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix
)

In [ ]:
train_df = pd.read_csv(
    "../data/processed/train_processed.csv"
)

train_df.shape

In [ ]:
train_df.head()

In [ ]:
X = train_df.drop(
    columns=["TARGET"]
)

y = train_df["TARGET"]

print("Feature Matrix Shape:", X.shape)
print("Target Shape:", y.shape)

## Train-Test Split

The dataset is divided into training and testing subsets.

Stratified sampling is used to preserve the original class distribution and ensure reliable model evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
print(
    y_train.value_counts(normalize=True) * 100
)

### Observations

The dataset exhibits significant class imbalance, with 91.93% non-default cases and 8.07% default cases.

To maintain this distribution during model development, stratified sampling was applied when creating the training and testing datasets.

Since a model could achieve high accuracy by simply predicting the majority class, evaluation will focus on ROC-AUC and other classification metrics that better measure the ability to distinguish between defaulters and non-defaulters.

This approach aligns with industry practices in credit risk modeling, where identifying high-risk applicants is more important than maximizing overall accuracy.

## Baseline Model: Logistic Regression

Logistic Regression is used as a baseline classification model.

Because Logistic Regression is sensitive to feature scale, numerical features are standardized before training.

The baseline model provides a reference point against which more advanced algorithms such as XGBoost and LightGBM can be compared.

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

print(X_train_scaled.shape)
print(X_test_scaled.shape)

In [ ]:
log_model = LogisticRegression(
    max_iter=1000,
    random_state=42
)

log_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
log_preds = log_model.predict(X_test_scaled)

log_probs = log_model.predict_proba(
    X_test_scaled
)[:, 1]

In [ ]:
log_auc = roc_auc_score(
    y_test,
    log_probs
)

print("Logistic Regression ROC-AUC:", log_auc)

In [ ]:
print(
    classification_report(
        y_test,
        log_preds
    )
)

In [ ]:
confusion_matrix(
    y_test,
    log_preds
)

## Addressing Class Imbalance

The baseline Logistic Regression model achieved a reasonable ROC-AUC score but demonstrated very poor recall for the default class.

To improve the model's ability to identify defaulters, class weighting is introduced. This assigns greater importance to minority-class observations during training and helps reduce bias toward the majority class.

In [ ]:
balanced_log_model = LogisticRegression(
    class_weight="balanced",
    max_iter=1000,
    random_state=42
)

balanced_log_model.fit(
    X_train_scaled,
    y_train
)

In [ ]:
balanced_preds = balanced_log_model.predict(
    X_test_scaled
)

balanced_probs = (
    balanced_log_model.predict_proba(
        X_test_scaled
    )[:, 1]
)

In [ ]:
balanced_auc = roc_auc_score(
    y_test,
    balanced_probs
)

print("Balanced Logistic Regression ROC-AUC:", balanced_auc)

In [ ]:
print(
    classification_report(
        y_test,
        balanced_preds
    )
)

In [ ]:
confusion_matrix(
    y_test,
    balanced_preds
)

## Logistic Regression Evaluation Summary

Two Logistic Regression models were evaluated.

The baseline model achieved a ROC-AUC score of approximately 0.75 but identified very few default cases due to the strong class imbalance present in the dataset.

Introducing class weighting substantially improved recall for the default class (from 1% to 68%), demonstrating the importance of imbalance-aware training strategies. While precision decreased, the model became significantly more effective at identifying potentially risky applicants.

These results establish a strong baseline and motivate the use of more advanced ensemble methods such as XGBoost and LightGBM.

## Logistic Regression Model Comparison

| Metric | Logistic Regression | Balanced Logistic Regression |
|----------|----------:|----------:|
| ROC-AUC | 0.7486 | 0.7482 |
| Accuracy | 0.92 | 0.69 |
| Precision (Default Class) | 0.58 | 0.16 |
| Recall (Default Class) | 0.01 | 0.68 |
| F1-Score (Default Class) | 0.02 | 0.26 |
| True Positives | 53 | 3,355 |
| False Negatives | 4,912 | 1,610 |
| False Positives | 39 | 17,568 |

### Key Findings

- The baseline Logistic Regression model achieved a reasonable ROC-AUC score but failed to identify most default cases.
- Applying class balancing dramatically improved recall for the default class, increasing it from 1% to 68%.
- The balanced model successfully identified 3,355 default cases compared to only 53 identified by the baseline model.
- This improvement came at the cost of lower precision and a larger number of false positives.
- ROC-AUC remained almost unchanged, indicating that class balancing primarily affected the decision threshold rather than the model's underlying ranking ability.
- Since credit risk prediction prioritizes identifying risky applicants, the balanced model provides a more useful baseline despite its lower precision.

## XGBoost Model Development

While Logistic Regression provided a useful baseline, it assumes a linear relationship between features and the target variable.

XGBoost is a gradient boosting algorithm capable of capturing complex nonlinear patterns and feature interactions, making it particularly effective for structured financial datasets.

The objective of this stage is to determine whether a more sophisticated ensemble model can improve discriminatory power beyond the Logistic Regression baseline.

In [ ]:
from xgboost import XGBClassifier

In [ ]:
negative = (y_train == 0).sum()
positive = (y_train == 1).sum()

scale_pos_weight = negative / positive

print(scale_pos_weight)

In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric="logloss"
)

xgb_model.fit(
    X_train,
    y_train
)

In [ ]:
xgb_preds = xgb_model.predict(
    X_test
)

xgb_probs = (
    xgb_model.predict_proba(
        X_test
    )[:, 1]
)

In [ ]:
xgb_auc = roc_auc_score(
    y_test,
    xgb_probs
)

print("XGBoost ROC-AUC:", xgb_auc)

In [ ]:
print(
    classification_report(
        y_test,
        xgb_preds
    )
)

In [ ]:
confusion_matrix(
    y_test,
    xgb_preds
)

In [ ]:
feature_importance = pd.Series(
    xgb_model.feature_importances_,
    index=X_train.columns
)

feature_importance.sort_values(
    ascending=False
).head(20)

## XGBoost Evaluation Summary

XGBoost achieved the strongest performance observed so far, increasing ROC-AUC from 0.7486 (Logistic Regression) to 0.7595.

The model maintained strong recall for the default class while improving overall discriminatory power and F1-score.

Feature importance analysis revealed that several engineered variables, including EXT_SOURCE_MEAN, EMPLOYMENT_YEARS, EXT_SOURCE_1_MISSING, OWN_CAR_AGE_MISSING, and DAYS_EMPLOYED_PLACEHOLDER, contributed significantly to prediction performance.

These findings validate the feature engineering decisions made during preprocessing and demonstrate the effectiveness of gradient boosting methods for credit risk prediction.

## Model Performance Comparison

| Metric | Logistic Regression | Balanced Logistic Regression | XGBoost |
|----------|----------:|----------:|----------:|
| ROC-AUC | 0.7486 | 0.7482 | **0.7595** |
| Accuracy | **0.92** | 0.69 | 0.72 |
| Precision (Default Class) | **0.58** | 0.16 | 0.17 |
| Recall (Default Class) | 0.01 | **0.68** | 0.66 |
| F1-Score (Default Class) | 0.02 | 0.26 | **0.28** |

### Key Findings

- Logistic Regression achieved the highest accuracy, but largely failed to identify default cases due to severe class imbalance.
- Applying class balancing dramatically improved recall, increasing the model's ability to detect defaulters.
- XGBoost delivered the best overall performance, achieving the highest ROC-AUC and F1-score while maintaining strong recall.
- The improvement in ROC-AUC indicates that XGBoost is better at distinguishing between risky and non-risky applicants.
- Feature importance analysis further confirmed the value of engineered features created during preprocessing.

## LightGBM Model Development

LightGBM is a gradient boosting framework specifically designed for efficient training on large structured datasets.

It uses a leaf-wise tree growth strategy that often produces strong predictive performance while maintaining fast training speed.

Since LightGBM has demonstrated excellent results in many credit risk and Kaggle competitions, it is evaluated alongside XGBoost to determine the best-performing model for this project.

In [ ]:
from lightgbm import LGBMClassifier

In [ ]:
lgbm_model = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42
)

lgbm_model.fit(
    X_train,
    y_train
)

In [ ]:
lgbm_preds = lgbm_model.predict(
    X_test
)

lgbm_probs = lgbm_model.predict_proba(
    X_test
)[:, 1]

In [ ]:
lgbm_auc = roc_auc_score(
    y_test,
    lgbm_probs
)

print("LightGBM ROC-AUC:", lgbm_auc)

In [ ]:
print(
    classification_report(
        y_test,
        lgbm_preds
    )
)

In [ ]:
confusion_matrix(
    y_test,
    lgbm_preds
)

In [ ]:
lgbm_importance = pd.Series(
    lgbm_model.feature_importances_,
    index=X_train.columns
)

lgbm_importance.sort_values(
    ascending=False
).head(20)

## Feature Validation: Removing Identifier Variables

During LightGBM feature importance analysis, the variable `SK_ID_CURR` appeared among the most influential features. Since this column is a unique customer identifier rather than a meaningful customer attribute, its predictive contribution may reflect accidental patterns in the dataset rather than genuine indicators of credit risk.

Identifier variables generally do not contain business-relevant information and can introduce spurious correlations that reduce model interpretability and generalizability. Therefore, `SK_ID_CURR` will be removed from the feature set and the model will be retrained.

The objective of this experiment is to evaluate the impact of the identifier on model performance and verify that predictive power is primarily driven by legitimate customer characteristics and engineered features. If performance remains stable after removal, it will provide stronger evidence that the model has learned meaningful credit-risk patterns rather than relying on dataset-specific artifacts.

In [ ]:
"SK_ID_CURR" in X_train.columns

In [ ]:
X_train_no_id = X_train.drop(
    columns=["SK_ID_CURR"]
)

X_test_no_id = X_test.drop(
    columns=["SK_ID_CURR"]
)

print(X_train_no_id.shape)
print(X_test_no_id.shape)

In [ ]:
lgbm_model_no_id = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    class_weight="balanced",
    random_state=42
)

lgbm_model_no_id.fit(
    X_train_no_id,
    y_train
)

In [ ]:
lgbm_preds_no_id = lgbm_model_no_id.predict(
    X_test_no_id
)

lgbm_probs_no_id = (
    lgbm_model_no_id.predict_proba(
        X_test_no_id
    )[:, 1]
)

In [ ]:
lgbm_auc_no_id = roc_auc_score(
    y_test,
    lgbm_probs_no_id
)

print(
    "LightGBM ROC-AUC (No ID):",
    lgbm_auc_no_id
)

In [ ]:
print(
    classification_report(
        y_test,
        lgbm_preds_no_id
    )
)

In [ ]:
confusion_matrix(
    y_test,
    lgbm_preds_no_id
)

In [ ]:
importance_no_id = pd.Series(
    lgbm_model_no_id.feature_importances_,
    index=X_train_no_id.columns
)

importance_no_id.sort_values(
    ascending=False
).head(20)

### Results After Removing Identifier Variable

After removing `SK_ID_CURR` and retraining LightGBM, model performance remained stable and showed a slight improvement in ROC-AUC.

| Metric | Before Removal | After Removal |
|----------|----------:|----------:|
| ROC-AUC | 0.7596 | 0.7600 |
| Recall | 0.67 | 0.67 |
| Accuracy | 0.72 | 0.71 |

The negligible change in performance indicates that the model's predictive capability is primarily driven by meaningful customer attributes and engineered features rather than the customer identifier.

This experiment improves model interpretability and aligns the feature set with standard machine learning best practices by excluding non-informative identifier variables.

## XGBoost Identifier Validation

Since the customer identifier (`SK_ID_CURR`) was found to contribute to LightGBM feature importance, the same validation procedure is performed for XGBoost.

The objective is to determine whether the identifier influences model performance and to ensure a fair comparison between the two leading gradient boosting models using an identical feature set.

If XGBoost maintains or improves performance after removing the identifier, the comparison between XGBoost and LightGBM will be based solely on meaningful customer attributes and engineered features.

In [ ]:
xgb_model_no_id = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=6,
    scale_pos_weight=11,
    random_state=42
)

xgb_model_no_id.fit(
    X_train_no_id,
    y_train
)

In [ ]:
xgb_preds_no_id = xgb_model_no_id.predict(
    X_test_no_id
)

xgb_probs_no_id = (
    xgb_model_no_id.predict_proba(
        X_test_no_id
    )[:, 1]
)

In [ ]:
xgb_auc_no_id = roc_auc_score(
    y_test,
    xgb_probs_no_id
)

print(
    "XGBoost ROC-AUC (No ID):",
    xgb_auc_no_id
)

In [ ]:
print(
    classification_report(
        y_test,
        xgb_preds_no_id
    )
)

In [ ]:
confusion_matrix(
    y_test,
    xgb_preds_no_id
)

In [ ]:
xgb_importance_no_id = pd.Series(
    xgb_model_no_id.feature_importances_,
    index=X_train_no_id.columns
)

xgb_importance_no_id.sort_values(
    ascending=False
).head(20)

### XGBoost Identifier Validation Results

After removing `SK_ID_CURR` and retraining XGBoost, the ROC-AUC score decreased slightly from 0.7595 to 0.7586.

The small performance difference suggests that the identifier contained a limited amount of dataset-specific signal, but it was not a major contributor to predictive performance. The model continued to achieve strong results using only meaningful customer attributes and engineered features.

Compared to XGBoost, the identifier-free LightGBM model achieved superior ROC-AUC performance and was therefore selected as the primary candidate for further optimization and deployment.

## Ensemble Modeling: LightGBM + XGBoost

Ensemble methods combine predictions from multiple models to improve robustness and predictive performance. Since LightGBM and XGBoost achieved the strongest results among all evaluated models, an ensemble of their predicted probabilities is constructed.

The intuition behind this approach is that the two models may capture different patterns within the data and make different prediction errors. By averaging their probability estimates, the ensemble can potentially reduce variance and improve overall discrimination between default and non-default applicants.

The ensemble performance will be compared against the individual models to determine whether combining the models provides a measurable improvement in credit-risk prediction.

In [ ]:
ensemble_probs = (
    lgbm_probs_no_id +
    xgb_probs_no_id
) / 2

In [ ]:
ensemble_auc = roc_auc_score(
    y_test,
    ensemble_probs
)

print(
    "Ensemble ROC-AUC:",
    ensemble_auc
)

In [ ]:
ensemble_preds = (
    ensemble_probs >= 0.50
).astype(int)

In [ ]:
print(
    classification_report(
        y_test,
        ensemble_preds
    )
)

In [ ]:
confusion_matrix(
    y_test,
    ensemble_preds
)

In [ ]:
comparison = pd.DataFrame({
    "Model": [
        "XGBoost",
        "LightGBM",
        "LightGBM (No ID)",
        "XGBoost (No ID)",
        "Ensemble"
    ],
    "ROC_AUC": [
        xgb_auc,
        lgbm_auc,
        lgbm_auc_no_id,
        xgb_auc_no_id,
        ensemble_auc
    ]
})

comparison.sort_values(
    "ROC_AUC",
    ascending=False
)

# Final Model Performance Comparison

The following models were evaluated throughout the project to identify the most effective approach for predicting loan default risk. Performance was assessed primarily using **ROC-AUC**, which measures a model's ability to distinguish between defaulters and non-defaulters across all classification thresholds. Additional metrics such as precision, recall, and F1-score for the default class were also considered due to the highly imbalanced nature of the dataset.

| Model | ROC-AUC | Precision | Recall | F1-Score |
|---------|---------:|---------:|---------:|---------:|
| Logistic Regression | 0.7486 | 0.58 | 0.01 | 0.02 |
| Balanced Logistic Regression | 0.7482 | 0.16 | 0.68 | 0.26 |
| XGBoost | 0.7595 | 0.17 | 0.66 | 0.28 |
| LightGBM | 0.7596 | 0.17 | 0.67 | 0.28 |
| XGBoost (Without SK_ID_CURR) | 0.7586 | 0.18 | 0.64 | 0.28 |
| LightGBM (Without SK_ID_CURR) | 0.7600 | 0.17 | 0.67 | 0.27 |
| **LightGBM + XGBoost Ensemble** | **0.7604** | **0.18** | **0.66** | **0.28** |

## Model Development Summary

### Logistic Regression Baseline
A standard Logistic Regression model was used as the initial benchmark. While it achieved a reasonable ROC-AUC score, it failed to identify default cases effectively due to the severe class imbalance present in the dataset.

### Balanced Logistic Regression
Applying class weighting significantly improved recall for the minority class, enabling the model to detect a much larger proportion of defaulters. This highlighted the importance of addressing class imbalance in credit-risk prediction problems.

### Gradient Boosting Models
XGBoost and LightGBM were subsequently trained to capture complex non-linear relationships between applicant characteristics and loan default behaviour. Both models outperformed the logistic regression baselines and demonstrated substantially stronger predictive capability.

### Feature Engineering Impact
Several engineered features emerged as highly influential predictors, including:

- `EXT_SOURCE_MEAN`
- `AGE_YEARS`
- `EMPLOYMENT_YEARS`
- `EXT_SOURCE_1_MISSING`
- `OWN_CAR_AGE_MISSING`
- `ORGANIZATION_TYPE_FREQ`

The importance of these variables confirms that feature engineering and missing-value treatment contributed meaningfully to model performance.

### Identifier Validation
Feature importance analysis revealed that `SK_ID_CURR` appeared among the influential features in some models. Since customer identifiers should not contain predictive information, additional validation experiments were conducted.

Both LightGBM and XGBoost were retrained after removing the identifier. Performance remained largely unchanged, confirming that the models were learning from meaningful customer attributes rather than relying on dataset-specific identifiers.

### Ensemble Modeling
Finally, a probability-averaging ensemble combining LightGBM and XGBoost was constructed. The ensemble achieved the highest ROC-AUC score among all evaluated approaches while maintaining strong recall for the default class.

---

# Final Selected Model

## LightGBM + XGBoost Ensemble

### Performance

| Metric | Value |
|----------|----------:|
| ROC-AUC | **0.7604** |
| Precision | **0.18** |
| Recall | **0.66** |
| F1-Score | **0.28** |

### Reason for Selection

The LightGBM + XGBoost Ensemble was selected as the final model because it:

- Achieved the highest ROC-AUC score among all evaluated models.
- Combined the strengths of the two best-performing gradient boosting algorithms.
- Maintained strong default-detection capability on an imbalanced dataset.
- Demonstrated robustness after removal of identifier-based features.
- Leveraged engineered features and domain-informed preprocessing strategies developed throughout the project.
- Represents a practical and production-ready approach for credit-risk prediction.

### Conclusion

The final ensemble model successfully balances predictive performance, robustness, and interpretability, making it the strongest candidate for deployment within the CreditWise AI credit-risk assessment system.

## Threshold Optimization

The ensemble model produces probabilities representing the likelihood of customer default. By default, a probability threshold of 0.50 is used to convert these probabilities into class predictions.

However, in credit-risk applications, the optimal threshold may differ from 0.50 because the cost of approving a high-risk applicant is often greater than the cost of rejecting a low-risk applicant.

To identify a more suitable decision boundary, the ensemble model will be evaluated across multiple probability thresholds. The resulting precision, recall, and F1-score values will be compared to understand the trade-offs between identifying potential defaulters and minimizing false alarms.

This analysis helps translate model predictions into practical business decisions and supports the selection of an operational threshold for deployment.

In [ ]:
threshold_results = []

for threshold in [0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60]:

    preds = (ensemble_probs >= threshold).astype(int)

    report = classification_report(
        y_test,
        preds,
        output_dict=True
    )

    threshold_results.append({
        "Threshold": threshold,
        "Precision": report["1"]["precision"],
        "Recall": report["1"]["recall"],
        "F1": report["1"]["f1-score"]
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df.sort_values(
    by="F1",
    ascending=False
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Precision"],
    marker="o",
    label="Precision"
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["Recall"],
    marker="o",
    label="Recall"
)

plt.plot(
    threshold_df["Threshold"],
    threshold_df["F1"],
    marker="o",
    label="F1 Score"
)

plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Threshold Optimization")
plt.legend()

plt.show()

## Threshold Optimization Results

The ensemble model was evaluated across multiple probability thresholds to determine an appropriate decision boundary for credit-risk prediction. Since the cost of misclassification differs between lending scenarios, multiple threshold options were analyzed rather than relying solely on the default threshold of 0.50.

| Threshold | Precision | Recall | F1 Score |
|------------|-----------:|--------:|---------:|
| 0.30 | 0.118 | 0.883 | 0.208 |
| 0.35 | 0.131 | 0.837 | 0.227 |
| 0.40 | 0.145 | 0.781 | 0.244 |
| 0.45 | 0.160 | 0.722 | 0.261 |
| 0.50 | 0.176 | 0.655 | 0.277 |
| 0.55 | 0.195 | 0.584 | 0.293 |
| 0.60 | 0.216 | 0.503 | 0.302 |

### Observations

- As the classification threshold increased, precision improved while recall decreased.
- Lower thresholds identified a larger proportion of defaulters but generated a higher number of false positives.
- Higher thresholds produced more conservative predictions, reducing false alarms at the cost of missing more defaulters.
- The highest F1-score was achieved at a threshold of **0.60**, indicating the strongest balance between precision and recall from a purely statistical perspective.

### Threshold Analysis

#### Threshold = 0.50 (Risk-Sensitive Lending Strategy)

| Metric | Value |
|---------|---------:|
| Precision | 0.176 |
| Recall | 0.655 |
| F1 Score | 0.277 |

Advantages:
- Detects the largest proportion of potential defaulters among the shortlisted thresholds.
- Minimizes the risk of approving high-risk borrowers.
- Suitable when the cost of loan default is significantly higher than the cost of rejecting a creditworthy applicant.

Disadvantages:
- Produces more false positives.
- May unnecessarily reject a larger number of low-risk applicants.

---

#### Threshold = 0.55 (Balanced Operational Strategy)

| Metric | Value |
|---------|---------:|
| Precision | 0.195 |
| Recall | 0.584 |
| F1 Score | 0.293 |

Advantages:
- Provides a balanced tradeoff between precision and recall.
- Improves precision compared to 0.50 while retaining a substantial portion of default detection capability.
- Represents a practical compromise between portfolio growth and risk management.

Disadvantages:
- Misses more defaulters than the 0.50 threshold.
- Does not achieve the maximum F1-score.

---

#### Threshold = 0.60 (Performance-Optimized Strategy)

| Metric | Value |
|---------|---------:|
| Precision | 0.216 |
| Recall | 0.503 |
| F1 Score | 0.302 |

Advantages:
- Achieves the highest precision and highest F1-score.
- Produces the fewest false positives among the evaluated candidate thresholds.
- Suitable when reducing unnecessary loan rejections is a priority.

Disadvantages:
- Misses a significantly larger proportion of potential defaulters.
- May expose the lender to greater credit risk due to reduced recall.

### Final Recommendation

The threshold optimization exercise demonstrates that there is no universally optimal threshold; the choice depends on business objectives and risk appetite.

- **0.50** is preferred for risk-sensitive lending environments where maximizing default detection is the primary objective.
- **0.55** provides the most balanced operational tradeoff between risk control and customer approval rates.
- **0.60** offers the strongest statistical performance through the highest F1-score and precision.

For CreditWise AI, **0.55 is chosen as the operational threshold**, as it provides a balanced compromise between identifying potential defaulters and minimizing unnecessary rejections while maintaining strong overall model performance.

### Final Deployment Configuration

- **Model:** LightGBM + XGBoost Probability-Averaging Ensemble
- **ROC-AUC:** 0.7604
- **Recommended Threshold:** 0.55
- **Use Case:** Credit Default Risk Assessment

# Notebook Conclusion

This notebook focused on model development, evaluation, and optimization for the CreditWise AI credit-risk prediction system.

## Models Evaluated

- Logistic Regression
- Balanced Logistic Regression
- XGBoost
- LightGBM
- LightGBM + XGBoost Ensemble

## Key Findings

- Class imbalance significantly affected baseline model performance.
- Gradient boosting models substantially outperformed logistic regression approaches.
- Feature engineering and missing-value treatment contributed meaningfully to predictive performance.
- Identifier validation confirmed that model predictions were not dependent on customer ID variables.
- Ensemble learning achieved the highest ROC-AUC score among all evaluated approaches.
- Threshold optimization demonstrated the trade-off between default detection and false-positive rates.

## Final Selected Configuration

| Component | Selection |
|------------|------------|
| Model | LightGBM + XGBoost Ensemble |
| ROC-AUC | 0.7604 |
| Threshold | 0.55 |
| Strategy | Balanced Credit-Risk Assessment |

## Next Steps

The next phase of the project will focus on model persistence, prediction pipelines, and deployment of the CreditWise AI system through an interactive user interface.

## Preserving the Benchmark Test Dataset

The holdout test dataset used during model evaluation is preserved for future benchmarking.

Retaining the same test set enables fair comparison of future model improvements, feature engineering techniques, hyperparameter tuning experiments, and deployment versions.

This approach follows industry best practices by maintaining a consistent evaluation benchmark throughout the machine learning lifecycle.

In [ ]:
import os

os.makedirs("../data/processed", exist_ok=True)

X_test.to_csv(
    "../data/processed/X_test.csv",
    index=False
)

y_test.to_csv(
    "../data/processed/y_test.csv",
    index=False
)

print("Benchmark test datasets saved successfully")

In [ ]:
X_test_check = pd.read_csv(
    "../data/processed/X_test.csv"
)

y_test_check = pd.read_csv(
    "../data/processed/y_test.csv"
)

print(X_test_check.shape)
print(y_test_check.shape)